# 📡 Data Drift & Model Monitoring — Evidently AI

This notebook demonstrates the monitoring layer of the MLOps pipeline.

- Detect **data drift** between training (reference) and production (current) data
- Track **model performance degradation** over time
- Generate alerts when retraining is needed

**Designed to be shown during interviews — demonstrates real MLOps thinking.**

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
import json
from pathlib import Path

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

print('✅ Setup complete')

## 1. Simulate Drift Scenario

In [ ]:
# Load reference (training snapshot) and current (production) data
ref_path = f"../{config['paths']['reference_data']}"
cur_path = f"../{config['paths']['features_data']}"

if not Path(ref_path).exists():
    print('⚠️  Run the pipeline first: python scripts/run_pipeline.py')
else:
    reference_df = pd.read_parquet(ref_path)
    current_df   = pd.read_parquet(cur_path)

    print(f'Reference shape : {reference_df.shape}')
    print(f'Current shape   : {current_df.shape}')

    # ── Simulate drift by perturbing current data ────────────────
    np.random.seed(42)
    current_drifted = current_df.copy()

    # Artificially shift MonthlyCharges up by 20% (simulates price increase)
    if 'MonthlyCharges' in current_drifted.columns:
        current_drifted['MonthlyCharges'] *= 1.20

    # Shift tenure distribution (new customer acquisition spike)
    if 'tenure' in current_drifted.columns:
        current_drifted['tenure'] = current_drifted['tenure'] * 0.6

    print('\n✅ Drift simulation applied:')
    print('  • MonthlyCharges shifted up +20%  (price increase scenario)')
    print('  • tenure shifted down 40%          (new customer surge scenario)')

## 2. Visual: Reference vs Drifted Distributions

In [ ]:
if Path(ref_path).exists():
    features_to_plot = ['MonthlyCharges', 'tenure', 'risk_score', 'charge_ratio']
    features_to_plot = [f for f in features_to_plot if f in reference_df.columns]

    fig = make_subplots(
        rows=1, cols=len(features_to_plot),
        subplot_titles=features_to_plot
    )

    for i, feat in enumerate(features_to_plot, 1):
        fig.add_trace(go.Histogram(
            x=reference_df[feat], name='Reference',
            opacity=0.65, marker_color='#6366f1',
            histnorm='probability density',
            showlegend=(i == 1)
        ), row=1, col=i)

        fig.add_trace(go.Histogram(
            x=current_drifted[feat], name='Current (Drifted)',
            opacity=0.65, marker_color='#ef4444',
            histnorm='probability density',
            showlegend=(i == 1)
        ), row=1, col=i)

    fig.update_layout(
        barmode='overlay',
        title='Reference vs Drifted Feature Distributions',
        height=350,
        plot_bgcolor='rgba(0,0,0,0)'
    )
    fig.show()

    print('📌 Purple = training distribution | Red = current (drifted) distribution')
    print('   Clear shift visible in MonthlyCharges and tenure — would trigger drift alert')

## 3. Run Evidently Drift Report

In [ ]:
if Path(ref_path).exists():
    from src.monitoring.drift_monitor import DriftMonitor

    monitor = DriftMonitor(config)

    print('Running Evidently report on ORIGINAL data (no drift) ...')
    report_clean = monitor.run_full_report(reference_df, current_df)

    print('\nRunning Evidently report on DRIFTED data ...')
    report_drifted = monitor.run_full_report(reference_df, current_drifted)

    # Compare
    clean_ds   = report_clean.get('drift_summary',   {})
    drifted_ds = report_drifted.get('drift_summary', {})

    comparison = pd.DataFrame({
        'Metric': ['Dataset Drift', 'Drifted Features', 'Share Drifted', 'All Tests Passed'],
        'Clean Data': [
            str(clean_ds.get('dataset_drift', False)),
            clean_ds.get('n_drifted_features', 0),
            f"{clean_ds.get('share_drifted', 0):.1%}",
            str(report_clean.get('test_results', {}).get('all_passed', True))
        ],
        'Drifted Data': [
            str(drifted_ds.get('dataset_drift', False)),
            drifted_ds.get('n_drifted_features', 0),
            f"{drifted_ds.get('share_drifted', 0):.1%}",
            str(report_drifted.get('test_results', {}).get('all_passed', True))
        ]
    })

    print('\n=== DRIFT COMPARISON ===')
    print(comparison.to_string(index=False))

    if report_drifted.get('alerts'):
        print('\n🚨 ALERTS on drifted data:')
        for a in report_drifted['alerts']:
            print(f"  [{a['level']}] {a['msg']}")

## 4. Why This Matters in Production

### The Real-World Drift Problem

```
Month 1:  Model trained on data where avg MonthlyCharges = $65
Month 3:  Company raises prices → avg MonthlyCharges = $78
          → Model sees patterns it was never trained on
          → Predictions become unreliable
          → Churn rate spikes but model shows 'Low Risk' ❌
```

### How This Pipeline Solves It

```
Every Monday at 04:00 UTC:
  1. Airflow runs churn_auto_retrain DAG
  2. Evaluates Production model on last week's data
  3. If F1 < 0.75 → triggers Optuna retraining automatically
  4. New model compared to current Production
  5. If new model is better → auto-promoted to Production
  6. Streamlit dashboard updated with new metrics
  7. Slack alert sent to MLOps team
```

**This is the difference between a notebook model and a production ML system.**